### Libraries

In [ ]:
import numpy as np
import pandas as pd
import polars as pl

### Geoapify Example Datasets

In [ ]:
geoapify_places = pd.read_json('../../data/geoapify_places_api_example.json')
geoapify_places_df = pd.json_normalize(geoapify_places['features'])
print(geoapify_places_df.info())

### Data Processing : Geoapify API Example


In [8]:
day_mapper = {
    'Mo': 'Monday',
    'Tu': 'Tuesday',
    'We': 'Wednesday',
    'Th': 'Thursday',
    'Fr': 'Friday',
    'Sa': 'Saturday',
    'Su': 'Sunday',
}
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
# Helper Function
def format_date(data:str) -> str:
    delimitter = ';'
    if delimitter in data:
        date_list = data.split(delimitter)
    else:
        date = data + ';'
        date_list = date.split(delimitter)
    #split day from time
    date_list = [date.split(' ') for date in date_list]
    # eliminate spaces
    date_list = [list(filter(None, date)) for date in date_list]
    # format days : Mo -> Monday
    if len(date_list) > 1:
        date_list = [expand_date(date) for date in date_list]
    else:
        date_list = expand_date(date_list)
    if len(date_list) == 4:
        date_list = date_list[0] + date_list[1] + date_list[2] + date_list[3]
    if len(date_list) == 3:
        date_list = date_list[0] + date_list[1] + date_list[2]
    if len(date_list) == 2:
        date_list = date_list[0] + date_list[1]
    date_list = dict(date_list)

    result = str(date_list)

    return result

def expand_date(date:list[str]) -> list:
    result = []
    if len(date) != 2:
        return result
    day_part, time_part = date[0],date[1]
    if '-' in day_part and len(day_part) > 2:
        start_day, end_day = day_part.split('-')
        start_full = day_mapper[start_day]
        end_full = day_mapper[end_day]
        if start_full and end_full:
            start_idx = day_order.index(start_full)
            end_idx = day_order.index(end_full)
            for day in day_order[start_idx:end_idx+1]:
                result.append([day, time_part])
    else:
        # Single day
        full_day = day_mapper[day_part]
        if full_day:
            result.append([full_day, time_part])
    return result

In [ ]:
# test cases
expand_test = format_date('Mo-Fr 07:00-20:00; Sa 09:00-18:00; Su 12:00-18:00')
print(expand_test)
expand_test_two = format_date('Mo-Fr 07:00-20:00')
print(expand_test_two)
expand_test_three = format_date('Mo-Fr 09:00-17:00; Sa off; Su off')
print(expand_test_three)

In [9]:
places_relevant_columns = ['properties.place_id','properties.name','properties.categories', 'properties.opening_hours','properties.country','properties.state','properties.city'
                        ,'properties.street','properties.postcode', 
                        'properties.lon','properties.lat']

places_rename ={
    'properties.place_id':'place_id',
    'properties.name': 'name',
    'properties.categories': 'categories',
    'properties.country': 'country',
    'properties.state': 'state',
    'properties.city': 'city',
    'properties.street': 'street',
    'properties.postcode': 'postcode',
    'properties.opening_hours':'hours',
    'properties.lon' : 'longitude',
    'properties.lat' : 'latitude'
}

places_df_view = geoapify_places_df[places_relevant_columns].rename(columns=places_rename)
places_df_view['hours'] = places_df_view['hours'].apply(format_date)


### Processed Data: Geoapify Example

** Note: Geoapify API's rarely contain information on amenities, so we have to cut the feature

In [ ]:
print(places_df_view.info())
print(places_df_view.head())

### Yelp Business Dataset

In [ ]:
yelp_business = pd.read_json('../../data/yelp_academic_dataset_business.json', lines=True)
print(yelp_business.info())
print(yelp_business.head())

### Data Processing: Yelp Business Dataset

In [45]:
# helper function
def f_t(data):
    if data is not None:
        data = dict(data)
        for k,v in data.items():
            data[k] = e_t(v)
        return data

def e_t(data):
    result = []
    time_pair = data.split('-')
    for time_str in time_pair:
        hour, minute = time_str.split(':')
        formatted_time = f"{hour}:{minute.zfill(2)}"
        result.append(formatted_time)
    result_str = result[0] +'-'+ result[1]
    return result_str



In [ ]:
# test 
test_one = {'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30'}
result_one = f_t(test_one)
print(result_one)

In [46]:
yelp_business = yelp_business.assign(country = 'United States')
yelp_business_relevant_columns = ['business_id','name','categories','hours','country','state','city','address','postal_code',
        'latitude','longitude']
yelp_business_rename = {
        'business_id':'place_id',
        'address' : 'street',
        'postal_code':'postcode',
}
yelp_business['categories'] = yelp_business['categories'].str.split(',')
yelp_business['hours'] = yelp_business['hours'].apply(f_t)
yelp_business_df_view = yelp_business[yelp_business_relevant_columns].rename(columns=yelp_business_rename)

### Processed Data: Yelp Business data

In [ ]:
print(yelp_business_df_view.info())
print(yelp_business_df_view.head())

### Yelp User dataset
* Note: dataset is too large to be uploaded on github
[Dataset Link](https://business.yelp.com/data/resources/open-dataset/)

- Assessment:

In [ ]:
# large dataset that needs polars instead of pandas
yelp_users = pl.read_ndjson('../../data/yelp_academic_dataset_user.json')
print(yelp_users.head(5))
print(yelp_users.columns)

### Melbourne visits dataset
- Assessment: Dataset's features aren't useful for our usecase

In [ ]:
Melbourne_user_visits = pd.read_csv('../../data/userVisits-Melb-allPOI.csv',sep=';')
print(Melbourne_user_visits.head())

### Melbourne Locations dataset

- Assessment: Too small of a data sample (242 rows of data)

In [ ]:
Melbourne_places = pd.read_csv('../../data/POI-Melb.csv',sep=';')
print(Melbourne_places.head())

### Google Reviews New York dataset
- Note: dataset is too large to be uploaded to github

In [ ]:
google_reviews_NYC = pl.read_ndjson('../../data/review-New_York_10.json')
columns = ['user_id','name','rating','text','gmap_id']
print(google_reviews_NYC[columns].head())

### User Preference Mock dataset

In [39]:
# data options
use_case = ['local','daytrip','travel','mixed']
group_type = ['solo','couple','friends','family','mixed']
daily_budget_tier = ['1','2','3','4']
trip_budget_tier = ['1','2','3','4']
preferred_tags = ['outdoor','cultural','food and drink','nightlife','shopping','wellness','historical','scenic','adventurous','family friendly','romantic','pet friendly','upscale','budget friendly']
exploration_score = ['1','2','3','4','5']
popularity_weight = ['1','2','3','4','5']
cuisine_prefrences = ['American','Italian','East Asian','Southeast Asian','South Asian','Mexican','Mediterranean','Vegetarian','Seafood']
dietary_restrictions = ['vegetarian','vegan','gluten-free','halal','kosher','nut allergy','dairy-free','none']
travel_mode = ['walking','biking','public transit','driving']
max_travel_minutes = ['< 10 minutes','10-20 minutes','20-40 minutes','> 40 minutes']
itinerary_pace =['packed','balanced','relaxed']

In [40]:
import random
def create_mock_user_preference():
    user_preference = {}
    user_preference['user_id'] = random.randint(1,100000)
    user_preference['use_case'] = random.choice(use_case)
    user_preference['group_type'] = random.choice(group_type)
    user_preference['daily_budget_tier'] = random.choice(daily_budget_tier)
    user_preference['trip_budget_tier'] = random.choice(trip_budget_tier)
    user_preference['preferred_tags'] = random.sample(preferred_tags,random.randint(1,3))
    user_preference['exploration_score'] = random.choice(exploration_score)
    user_preference['popularity_weight'] = random.choice(popularity_weight)
    user_preference['cuisine_prefrences'] = random.sample(cuisine_prefrences,random.randint(1,3))
    user_preference['dietary_restrictions'] = random.sample(dietary_restrictions,random.randint(1,3))
    user_preference['travel_mode'] = random.sample(travel_mode,random.randint(1,2))
    user_preference['max_travel_minutes'] = random.choice(max_travel_minutes)
    user_preference['itinerary_pace'] = random.choice(itinerary_pace)

    return user_preference

In [41]:
users = []
for _ in range(10000):
    users.append(create_mock_user_preference())

In [42]:
user_df = pd.DataFrame(users)

In [ ]:
print(user_df.head(10))

### Mock User Likes Dataset

In [44]:
# Data Options
user_id = list(user_df['user_id'].values)
place_id = list(places_df_view['place_id'].values)
liked = ['0','1']

In [45]:
def create_mock_user_likes():
    user_likes = {}
    user_likes['user_id'] = random.choice(user_id)
    user_likes['place_id'] = random.choice(place_id)
    user_likes['liked'] = random.choice(liked)
    user_likes['created_at'] = '16-03-2026'
    return user_likes

In [46]:
users_likes = []
for _ in range(1000):
    users_likes.append(create_mock_user_likes())

In [47]:
users_likes_df = pd.DataFrame(users_likes)

In [ ]:
print(users_likes_df.head(10))

## Data Schemas

### Proposed `User Likes` Schema
stores categories of what a user likes

| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| user_id | `string` | The ID of the user |
| place_id | `string` | The ID of the place from geoapify |
| Liked | `boolean` | If the user liked or disliked a place [0/1] |
| created_at | `DATETIME` | Time user placed the like/dislike |

### Proposed `User_Preference` schema
Stores information about user's preferences

| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| user_id | `string` | The ID of the user |
| use_case | `string` | How a user travels |
| part_type | `string` | Who a user travels with |
| daily_budget_tier | `string` | daily budget of the user |
| trip_budget_tier | `string` | user budget for trips |
| preferred_tags | `list[string]` | Actvities user preferres |
| exploration_score | `string` | How much the user likes to explore an area |
| popularity_weight | `string` | How much sentiment affects users choices |
| cuisines_prefrences | `list[string]` | Food user preferres |
| dietary_restriction | `list[string]` | user's dietary restrictions |
| travel_mode | `list[string]` | user's preferred mode of travel |
| max_travel_minutes | `string` | Maximum amount of miniutes a user wants to travel between locations |
| itinerary_pace | `string` | Maximum number of activities to schedule a day |

### Proposed `Users` schema
stores account information of the user

| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| user_id | `string` | The ID of the user |
| username | `string` | name of the user |
| email | `string` | the email associated with the user |
| password | `string` | hased password of the user |
| created_at | `DATETIME` | time user created their account |

### Proposed `Places` schema

Stores information about places
| Column Name | Data Type | Description |
| :--- | :--- | :--- |
| place_id | `string` | The ID of the location |
| name | `string` | The name of the location |
| categories | `list[string]` | The list of category tags associated with location |
| hours | `dictionary` | The list of days and hours a location is open for |
| country | `string` | The country the location is located at |
| state | `string` | The state the location is located at |
| city | `string` | The city the location is located at |
| street | `string` | The street the location is located at |
| postcode | `string` | The postal/zip code the location is located at |
| lat | `float` | The latitude of the location |
| lon | `float` | The longitude of the location |